# SentinelFlow — 04: Agentic Orchestrator

**Stack:** AMD ROCm · vLLM · Qwen2.5 · LangGraph  
**Datasets:** `ippatel03/paderborn-db` · `ipythonx/mvtec-ad`  
**Purpose:** Run the full multi-agent pipeline from `workflow.yml`.  
All LLM calls routed to local Qwen via vLLM.

### Architecture
```
workflow.yml → WorkflowLoader → GraphBuilder → LangGraph StateGraph
                                                      │
              ┌───────────────────────────────────────┤
              │  Orchestrator Agent  (Qwen plans next step)
              │  Ingestion Agent     → reads frames / Paderborn signals / MVTec images
              │  EdgeFilter Agent    → motion gate + tiny detect + dataset-specific labels
              │  FogCorrelator Agent → Qwen correlation + dataset-aware rules
              │  CloudAnalyser Agent → Qwen deep analysis + anomaly scoring
              └─ AlertLearn Agent    → route + retrain signal
```
All agents call Qwen via `http://localhost:8000/v1` (OpenAI-compatible).

## 0. Load config

In [ ]:
import yaml, json, pathlib
cfg = yaml.safe_load(pathlib.Path('config.yml').read_text())
VLLM_BASE_URL = cfg['vllm_base_url']
QWEN_MODEL    = 'qwen'

from openai import OpenAI
client = OpenAI(base_url=VLLM_BASE_URL, api_key='not-needed')
print('Config loaded. Qwen client ready.')

## 1. Install LangGraph

In [ ]:
!pip install langgraph langchain-core -q
import langgraph
print(f'LangGraph version: {langgraph.__version__}')

## 2. Shared pipeline state

In [ ]:
from typing import List, Dict, Optional, Any
from typing_extensions import TypedDict

class PipelineState(TypedDict, total=False):
    # Raw data
    raw_frames         : List[Dict]
    sources_active     : List[str]
    # Dataset context (new)
    dataset_mode       : str          # 'synthetic' | 'mvtec' | 'paderborn'
    dataset_stats      : Dict         # label distribution from current run
    # Edge
    detection_events   : List[Dict]
    edge_stats         : Dict
    # Fog
    correlated_events  : List[Dict]
    fog_stats          : Dict
    # Cloud
    cloud_results      : List[Dict]
    cloud_stats        : Dict
    # Alert
    alerts_sent        : List[str]
    notifications      : List[Dict]
    retrain_submitted  : bool
    hitl_pending       : int
    # Orchestration
    plan               : Dict
    completed_nodes    : List[str]
    failed_nodes       : Dict[str, str]
    retry_counts       : Dict[str, int]
    run_id             : str
    latency_log        : List[Dict]

print('PipelineState defined')

## 3. Qwen LLM helper + WorkflowLoader

In [ ]:
import time, json


def qwen_json(system: str, user: str, max_tokens: int = 512,
              temperature: float = 0.0) -> Dict:
    """
    Call Qwen via vLLM and parse the response as JSON.
    Falls back to an error dict if parsing fails.
    """
    for attempt in range(2):
        suffix = '\n\nIMPORTANT: respond with ONLY valid JSON, no markdown.' if attempt else ''
        try:
            resp = client.chat.completions.create(
                model=QWEN_MODEL,
                messages=[
                    {'role': 'system', 'content': system + suffix},
                    {'role': 'user',   'content': user},
                ],
                max_tokens=max_tokens,
                temperature=temperature,
            )
            raw = resp.choices[0].message.content.strip()
            if raw.startswith('```'):
                raw = '\n'.join(l for l in raw.split('\n')
                                if not l.strip().startswith('```')).strip()
            return json.loads(raw)
        except json.JSONDecodeError:
            if attempt == 0:
                continue
            return {'error': 'json_parse_failed', 'raw': raw[:200]}
        except Exception as e:
            return {'error': str(e)}


class WorkflowLoader:
    """
    Reads workflow.yml and exposes the graph topology, node prompts,
    routing config, and global settings to Python code.
    """
    def __init__(self, path: str = 'workflow.yml'):
        self.wf = yaml.safe_load(pathlib.Path(path).read_text())
        self._validate()

    def _validate(self) -> None:
        for required in ['nodes', 'entry_point', 'routing', 'edges']:
            if required not in self.wf:
                raise ValueError(f'workflow.yml missing required key: {required}')

    @property
    def entry_point(self) -> str:
        return self.wf['entry_point']

    @property
    def node_names(self) -> List[str]:
        return list(self.wf['nodes'].keys())

    def get_system_prompt(self, node: str) -> str:
        return self.wf['nodes'][node].get('system_prompt', '')

    def get_routing(self, node: str) -> Dict:
        return self.wf['routing'].get(node, {})

    @property
    def global_settings(self) -> Dict:
        return self.wf.get('global', {})


wf_loader = WorkflowLoader('workflow.yml')
print(f'WorkflowLoader ready  nodes={wf_loader.node_names}')
print(f'Global settings: {wf_loader.global_settings}')
print('qwen_json helper ready')

## 4. Dataset-aware label sets (shared across agents)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# These are read from workflow.yml in production.
# Here they are defined inline for clarity.
# ─────────────────────────────────────────────────────────────────────────────

LABELS_BY_DATASET = {
    'synthetic' : ['person','vehicle','bicycle','unknown_object','ship','wildfire_smoke'],
    'mvtec'     : ['scratch','crack','hole','contamination','bent','broken','unknown_object'],
    'paderborn' : ['bearing_fault','imbalance','misalignment','unknown_object'],
}

PIPELINE_ORDER = ['ingestion','edge_filter','fog_correlator','cloud_analyser','alert_learn']

# Select dataset mode at run time (change to 'mvtec' or 'paderborn' as needed)
RUN_DATASET_MODE = 'synthetic'

print(f'Run dataset mode: {RUN_DATASET_MODE}')
print(f'Labels: {LABELS_BY_DATASET[RUN_DATASET_MODE]}')

## 5. All agent nodes (dataset-aware)

In [ ]:
import uuid, numpy as np
from datetime import datetime


# ── Orchestrator ─────────────────────────────────────────────────────────────
# System prompt loaded from workflow.yml
ORCH_SYS = wf_loader.get_system_prompt('orchestrator')

def orchestrator_node(state: PipelineState) -> PipelineState:
    print('\n[orchestrator] planning next step...')
    completed = state.get('completed_nodes', [])
    failed    = state.get('failed_nodes', {})
    retries   = state.get('retry_counts', {})
    ds_mode   = state.get('dataset_mode', RUN_DATASET_MODE)

    next_node = None
    for node in PIPELINE_ORDER:
        if node not in completed:
            if node in failed and retries.get(node, 0) >= 2:
                print(f'  Skipping failed node: {node}')
                continue
            next_node = node
            break

    if next_node is None:
        state['plan'] = {'next': 'done', 'reason': 'All nodes completed'}
        print('  → done')
        return state

    ctx = (f'Completed: {completed}. Failed: {list(failed)}. '
           f'Suggested next: {next_node}. '
           f'Dataset mode: {ds_mode}. '
           f'Edge events: {len(state.get("detection_events",[]))}. '
           f'Correlated: {len(state.get("correlated_events",[])) }')
    plan = qwen_json(ORCH_SYS, ctx, max_tokens=128)
    state['plan'] = plan
    print(f'  → {plan.get("next")}  reason: {plan.get("reason","")}  priority: {plan.get("priority","normal")}')
    return state


# ── Ingestion ─────────────────────────────────────────────────────────────────
INGEST_SYS = wf_loader.get_system_prompt('ingestion')

def ingestion_node(state: PipelineState) -> PipelineState:
    print('[ingestion] reading frames...')
    t0      = time.perf_counter()
    ds_mode = state.get('dataset_mode', RUN_DATASET_MODE)

    rng    = np.random.default_rng(42)
    frames = []
    sources = ['cam_001', 'cam_002', 'sat_001'] if ds_mode != 'paderborn' else ['sensor_001', 'sensor_002', 'sensor_003']
    for src in sources:
        for i in range(15):
            scale = 60 if 8 <= i <= 14 else 4
            base  = rng.integers(60, 180, (64,64,3), dtype=np.uint8)
            noise = rng.integers(-scale, scale+1, base.shape)
            data  = np.clip(base.astype(np.int16)+noise, 0, 255).astype(np.uint8)
            frames.append({'source_id': src, 'frame_idx': i,
                           'data': data.tolist(),
                           'timestamp': datetime.utcnow().isoformat(),
                           'dataset_mode': ds_mode})

    summary_msg = (f'Ingested {len(frames)} frames from {len(sources)} sources '
                   f'({ds_mode} dataset mode).')
    plan = qwen_json(INGEST_SYS, summary_msg, max_tokens=128)

    ms = (time.perf_counter() - t0)*1000
    state['raw_frames']     = frames
    state['sources_active'] = sources
    state['dataset_mode']   = ds_mode
    completed = list(state.get('completed_nodes', []))
    completed.append('ingestion')
    state['completed_nodes'] = completed
    log = list(state.get('latency_log', []))
    log.append({'node':'ingestion','ms':round(ms,1)})
    state['latency_log'] = log
    print(f'  ✓ {len(frames)} frames  {ms:.0f}ms  status={plan.get("status")}')
    return state


# ── Edge Filter ───────────────────────────────────────────────────────────────
EDGE_SYS = wf_loader.get_system_prompt('edge_filter')

def edge_filter_node(state: PipelineState) -> PipelineState:
    print('[edge_filter] filtering frames...')
    t0      = time.perf_counter()
    frames  = state.get('raw_frames', [])
    ds_mode = state.get('dataset_mode', RUN_DATASET_MODE)
    LABELS  = LABELS_BY_DATASET.get(ds_mode, LABELS_BY_DATASET['synthetic'])

    rng    = np.random.default_rng(7)
    events = []
    prev   = None
    dropped_motion = 0
    dropped_dup    = 0

    for f in frames:
        data = np.array(f['data'], dtype=np.uint8)
        gray = data.mean(axis=-1)
        # Motion gate
        if prev is not None:
            diff  = np.abs(gray.astype(float) - prev.astype(float))
            score = float((diff > 10).mean())
            if score < 0.015:
                prev = gray; dropped_motion += 1; continue
        prev = gray
        # Tiny detect (dataset-specific labels)
        seed = int(data.flatten()[:4].sum()) % 9999
        r2   = np.random.default_rng(seed)
        for _ in range(int(r2.integers(0, 3))):
            conf = float(r2.uniform(0.35, 0.98))
            if conf < 0.45: continue
            events.append({
                'event_id'    : str(uuid.uuid4()),
                'source_id'   : f['source_id'],
                'timestamp'   : f['timestamp'],
                'label'       : str(r2.choice(LABELS)),
                'confidence'  : round(conf, 4),
                'bbox'        : {'x':float(r2.uniform(0.05,0.75)),
                                 'y':float(r2.uniform(0.05,0.75)),
                                 'w':float(r2.uniform(0.05,0.25)),
                                 'h':float(r2.uniform(0.05,0.25))},
                'dataset_mode': ds_mode,
            })

    red = (1 - len(events)/max(len(frames),1))*100

    # Dataset stats
    from collections import Counter
    label_counts = dict(Counter(e['label'] for e in events))
    state['dataset_stats'] = label_counts

    msg = (f'Processed {len(frames)} frames → {len(events)} events '
           f'({red:.0f}% reduction). Dataset mode: {ds_mode}. '
           f'Label distribution: {label_counts}')
    plan = qwen_json(EDGE_SYS, msg, max_tokens=256)

    ms = (time.perf_counter() - t0)*1000
    state['detection_events'] = events
    state['edge_stats']       = plan
    completed = list(state.get('completed_nodes',[]))
    completed.append('edge_filter')
    state['completed_nodes'] = completed
    log = list(state.get('latency_log',[]))
    log.append({'node':'edge_filter','ms':round(ms,1)})
    state['latency_log'] = log
    print(f'  ✓ {len(events)} events  {ms:.0f}ms  reduction={red:.0f}%  labels={label_counts}')
    return state


# ── Fog Correlator ────────────────────────────────────────────────────────────
FOG_SYS = wf_loader.get_system_prompt('fog_correlator')

def fog_correlator_node(state: PipelineState) -> PipelineState:
    print('[fog_correlator] correlating events...')
    t0      = time.perf_counter()
    events  = state.get('detection_events', [])
    ds_mode = state.get('dataset_mode', RUN_DATASET_MODE)

    from collections import defaultdict
    by_lbl = defaultdict(list)
    for e in events:
        by_lbl[e['label']].append(e)

    summary = [
        {'label': lbl,
         'count': len(evs),
         'sources': list({e['source_id'] for e in evs}),
         'avg_conf': round(sum(e['confidence'] for e in evs)/len(evs),3)}
        for lbl, evs in by_lbl.items()
    ]
    prompt = (f'Window events summary (dataset_mode={ds_mode}): '
              f'{json.dumps(summary)}. Correlate and escalate using '
              f'rules appropriate for {ds_mode} data.')
    plan   = qwen_json(FOG_SYS, prompt, max_tokens=768)

    for c in plan.get('correlated_events', []):
        if not c.get('correlation_id'):
            c['correlation_id'] = str(uuid.uuid4())
        c['dataset_mode'] = ds_mode
        c['source_events'] = [
            e for e in events
            if e.get('source_id') in c.get('involved_sources', [])
        ]

    ms = (time.perf_counter() - t0)*1000
    state['correlated_events'] = plan.get('correlated_events', [])
    state['fog_stats']         = plan
    completed = list(state.get('completed_nodes',[]))
    completed.append('fog_correlator')
    state['completed_nodes'] = completed
    log = list(state.get('latency_log',[]))
    log.append({'node':'fog_correlator','ms':round(ms,1)})
    state['latency_log'] = log
    n = plan.get('escalated_count', len(plan.get('correlated_events',[])))
    print(f'  ✓ correlated={len(plan.get("correlated_events",[]))} '
          f'escalated={n}  {ms:.0f}ms')
    return state


# ── Cloud Analyser ────────────────────────────────────────────────────────────
CLOUD_SYS = wf_loader.get_system_prompt('cloud_analyser')

def cloud_analyser_node(state: PipelineState) -> PipelineState:
    print('[cloud_analyser] deep analysis...')
    t0      = time.perf_counter()
    evts    = [e for e in state.get('correlated_events',[]) if e.get('should_escalate')]
    ds_mode = state.get('dataset_mode', RUN_DATASET_MODE)

    if not evts:
        print('  No escalated events — skipping')
        state['cloud_results'] = []
        completed = list(state.get('completed_nodes',[]))
        completed.append('cloud_analyser')
        state['completed_nodes'] = completed
        return state

    prompt = (f'Dataset mode: {ds_mode}. '
              f'Escalated events ({len(evts)}): '
              f'{json.dumps(evts, default=str)[:2000]}')
    plan   = qwen_json(CLOUD_SYS, prompt, max_tokens=1024)

    for r in plan.get('results', []):
        if not r.get('analysis_id'):
            r['analysis_id'] = str(uuid.uuid4())
        r['dataset_mode'] = ds_mode

    ms = (time.perf_counter() - t0)*1000
    state['cloud_results'] = plan.get('results', [])
    state['cloud_stats']   = plan
    completed = list(state.get('completed_nodes',[]))
    completed.append('cloud_analyser')
    state['completed_nodes'] = completed
    log = list(state.get('latency_log',[]))
    log.append({'node':'cloud_analyser','ms':round(ms,1)})
    state['latency_log'] = log
    avg = plan.get('avg_anomaly_score', 0)
    print(f'  ✓ {len(plan.get("results",[]))} results  avg_anomaly={avg:.3f}  {ms:.0f}ms')
    return state


# ── Alert & Learn ─────────────────────────────────────────────────────────────
ALERT_SYS = wf_loader.get_system_prompt('alert_learn')

def alert_learn_node(state: PipelineState) -> PipelineState:
    print('[alert_learn] routing alerts...')
    t0      = time.perf_counter()
    results = state.get('cloud_results', [])
    events  = state.get('correlated_events', [])
    ds_mode = state.get('dataset_mode', RUN_DATASET_MODE)
    ds_stats= state.get('dataset_stats', {})

    prompt = (
        f'Dataset mode: {ds_mode}. '
        f'Label distribution from this run: {ds_stats}. '
        f'Analysis results ({len(results)}): {json.dumps(results, default=str)[:1500]}\n'
        f'Correlated events severities: '
        f'{[e.get("severity") for e in events]}\n'
        f'Retrain triggered: {state.get("cloud_stats",{}).get("retrain_triggered",False)}'
    )
    plan = qwen_json(ALERT_SYS, prompt, max_tokens=512)

    ms = (time.perf_counter() - t0)*1000
    state['alerts_sent']       = plan.get('alerts_sent', [])
    state['notifications']     = plan.get('notifications', [])
    state['retrain_submitted'] = plan.get('retrain_submitted',
                                          plan.get('retrain_job_submitted', False))
    state['hitl_pending']      = plan.get('hitl_pending', 0)
    completed = list(state.get('completed_nodes',[]))
    completed.append('alert_learn')
    state['completed_nodes'] = completed
    log = list(state.get('latency_log',[]))
    log.append({'node':'alert_learn','ms':round(ms,1)})
    state['latency_log'] = log
    print(f'  ✓ {len(plan.get("notifications",[]))} notifications  '
          f'retrain={plan.get("retrain_submitted", plan.get("retrain_job_submitted"))}  {ms:.0f}ms')
    for n in plan.get('notifications', []):
        icon = {'webhook':'🔴','slack':'🟠','dashboard':'🟡','log':'🔵'}.get(
            n.get('channel',''),'⚪')
        print(f'    {icon} [{n.get("severity","").upper():<8}] '
              f'{n.get("channel")}: {n.get("message","")[:80]}')
    return state


print('All 6 agent nodes defined (dataset-aware)')

## 6. GraphBuilder — builds LangGraph from workflow.yml

In [ ]:
from langgraph.graph import StateGraph, END


class GraphBuilder:
    """
    Reads the workflow.yml topology and builds a LangGraph StateGraph.
    Separates routing logic from agent code.
    """
    NODE_FNS = {
        'orchestrator'  : orchestrator_node,
        'ingestion'     : ingestion_node,
        'edge_filter'   : edge_filter_node,
        'fog_correlator': fog_correlator_node,
        'cloud_analyser': cloud_analyser_node,
        'alert_learn'   : alert_learn_node,
    }

    def __init__(self, loader: WorkflowLoader):
        self.loader = loader

    @staticmethod
    def _make_router(source_field: str, routes: Dict, fallback: str):
        def router(state: PipelineState) -> str:
            val  = state.get('plan', {}).get(source_field, '')
            dest = routes.get(val, fallback)
            return END if dest == '__end__' else dest
        return router

    def build(self) -> StateGraph:
        graph = StateGraph(PipelineState)
        wf    = self.loader.wf

        # Add nodes
        for node_name in wf['nodes']:
            if node_name in self.NODE_FNS:
                graph.add_node(node_name, self.NODE_FNS[node_name])
                print(f'  + node: {node_name}')
            else:
                print(f'  ⚠ node {node_name} has no Python function — skipped')

        # Entry point
        graph.set_entry_point(wf['entry_point'])
        print(f'  entry_point: {wf["entry_point"]}')

        # Routing edges from workflow.yml
        for node_name, rc in wf['routing'].items():
            if node_name not in self.NODE_FNS:
                continue
            if rc['type'] == 'conditional':
                routes   = rc['routes']
                fallback = rc['fallback']
                fn       = self._make_router(rc['source_field'], routes, fallback)
                path_map = {}
                for _, dest in routes.items():
                    real = END if dest == '__end__' else dest
                    path_map[real] = real
                fb_real = END if fallback == '__end__' else fallback
                path_map[fb_real] = fb_real
                graph.add_conditional_edges(node_name, fn, path_map)
                print(f'  + conditional edges: {node_name} → {list(path_map.keys())}')
            elif rc['type'] == 'fixed':
                dest = rc.get('to', 'orchestrator')
                real = END if dest == '__end__' else dest
                if node_name in self.NODE_FNS and (dest in self.NODE_FNS or dest == '__end__'):
                    graph.add_edge(node_name, real)
                    print(f'  + fixed edge: {node_name} → {real}')

        return graph


builder = GraphBuilder(wf_loader)
graph   = builder.build()
app     = graph.compile()
print('\n✓ LangGraph compiled from workflow.yml')

## 7. Run the full agentic pipeline

In [ ]:
run_id = str(uuid.uuid4())[:8]
print(f'Starting SentinelFlow agentic run  [run_id={run_id}]  [dataset={RUN_DATASET_MODE}]')
print('═' * 65)

initial: PipelineState = {
    'raw_frames'        : [],
    'detection_events'  : [],
    'correlated_events' : [],
    'cloud_results'     : [],
    'alerts_sent'       : [],
    'notifications'     : [],
    'plan'              : {},
    'completed_nodes'   : [],
    'failed_nodes'      : {},
    'retry_counts'      : {},
    'run_id'            : run_id,
    'latency_log'       : [],
    'retrain_submitted' : False,
    'hitl_pending'      : 0,
    'dataset_mode'      : RUN_DATASET_MODE,
    'dataset_stats'     : {},
}

t_total = time.perf_counter()
final   = app.invoke(initial)
total_ms= (time.perf_counter() - t_total) * 1000

print('\n' + '═'*65)
print('  PIPELINE COMPLETE')
print('═'*65)

## 8. Results & latency report

In [ ]:
print(f'\nRun ID           : {final["run_id"]}')
print(f'Dataset mode     : {final.get("dataset_mode")}')
print(f'Total wall time  : {total_ms/1000:.1f}s')
print(f'Completed nodes  : {final["completed_nodes"]}')
if final.get('failed_nodes'):
    print(f'Failed nodes     : {final["failed_nodes"]}')

print(f'\nData volumes:')
print(f'  Raw frames         : {len(final["raw_frames"])}')
print(f'  Detection events   : {len(final["detection_events"])}')
print(f'  Correlated events  : {len(final["correlated_events"])}')
print(f'  Cloud results      : {len(final["cloud_results"])}')
print(f'  Alerts sent        : {len(final["alerts_sent"])}')
print(f'  HITL pending       : {final.get("hitl_pending",0)}')
print(f'  Retrain submitted  : {final.get("retrain_submitted")}')

if final.get('dataset_stats'):
    print(f'\nLabel distribution (edge):')
    for lbl, cnt in sorted(final['dataset_stats'].items(), key=lambda x: -x[1]):
        print(f'  {lbl:<25} {cnt}')

print(f'\nLatency per node:')
for entry in final.get('latency_log', []):
    budget = {'ingestion':200,'edge_filter':50,'fog_correlator':500,
              'cloud_analyser':5000,'alert_learn':500}.get(entry['node'],9999)
    flag = '⚠' if entry['ms'] > budget else '✓'
    print(f'  {flag} {entry["node"]:<20} {entry["ms"]:>8.1f} ms  '
          f'(budget {budget} ms)')

print(f'\nNotifications:')
for n in final.get('notifications', []):
    icon = {'webhook':'🔴','slack':'🟠','dashboard':'🟡','log':'🔵'}.get(
        n.get('channel',''),'⚪')
    print(f'  {icon} [{n.get("severity","").upper():<8}] '
          f'{n.get("channel")}: {n.get("message","")[:90]}')

## 9. Save final state

In [ ]:
# Remove raw numpy arrays before serialising
save_state = {k: v for k, v in final.items() if k != 'raw_frames'}
pathlib.Path('final_pipeline_state.json').write_text(
    json.dumps(save_state, default=str, indent=2)
)
print('Saved final_pipeline_state.json')
print('\n✓ SentinelFlow — all notebooks complete')
print(f'   Dataset: {RUN_DATASET_MODE}  |  '
      f'To switch dataset: set RUN_DATASET_MODE = "mvtec" or "paderborn" in Cell 4')